In [2]:
from langchain_community.vectorstores import OpenSearchVectorSearch
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Import OllamaEmbeddings for your vectors, and ChatOllama for the final RAG answer stage
from langchain_ollama import OllamaEmbeddings, ChatOllama


In [3]:
# 1. Configure OpenSearch Connection Details
OPENSEARCH_URL = "http://localhost:9200"
INDEX_NAME = "my-ollama-rag-index"
HTTP_AUTH = ("admin", "admin")  # Update with your credentials if required


In [4]:
# 2. Prepare and chunk your raw text
raw_document_text = """
OpenSearch features a vector database engine capable of handling 
k-NN similarity search, making it an excellent backend choice for 
local, private RAG workflows powered by Ollama.
"""

In [5]:

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_text(raw_document_text)
documents = [Document(page_content=chunk, metadata={"source": "local_guide.txt"}) for chunk in chunks]


In [6]:
 
# 3. Define the Ollama Embedding Model
# Ensure you have pulled this model locally via terminal first: `ollama pull nomic-embed-text`
embedding_model = OllamaEmbeddings(model="nomic-embed-text")

### Index creation 
If index does not exists create it 
```
curl -X PUT "http://localhost:9200/my-ollama-rag-index" \
     -H 'Content-Type: application/json' \
     -u admin:admin
```

### View Index 
```
{"acknowledged":true,"shards_acknowledged":true,"index":"my-ollama-rag-index"}
```
View the index using 
```
curl -X GET "http://localhost:9200/_cat/indices?v"
```
you should get a response : 


```
health status index               uuid                   pri rep docs.count docs.deleted store.size pri.store.size
yellow open   my-ollama-rag-index W_KrY3CnQDi5zWAdQoZ7IA   1   1          0            0       208b           208b
```


In [8]:
# 4. Insert documents and embeddings into OpenSearch
docsearch = OpenSearchVectorSearch.from_documents(
    documents=documents,
    embedding=embedding_model,
    opensearch_url=OPENSEARCH_URL,
    index_name=INDEX_NAME,
    http_auth=HTTP_AUTH,
    use_ssl=False,
    verify_certs=False,
    engine="nmslib",
    space_type="cosinesimil"
)

### View Documents in the index 

```
curl -X GET "http://localhost:9200/my-ollama-rag-index/_search?pretty" \
     -H 'Content-Type: application/json' \
     -d '{
       "size": 100,
       "query": {
         "match_all": {}
       }
     }'
```

### Ingestion Complete

In [9]:
# Force-create the index correctly using .from_documents()
docsearch = OpenSearchVectorSearch.from_documents(
    documents=documents,
    embedding=embedding_model,
    opensearch_url=OPENSEARCH_URL,
    index_name="my-ollama-rag-index",
    # THESE FOUR LINES ENFORCE THE KNN ENGINE:
    engine="nmslib",              
    space_type="cosinesimil",     
    index_params={"parameters": {"engine": "nmslib", "method": "hnsw"}},
    http_auth=HTTP_AUTH
)


In [ ]:

 




print(f"Successfully inserted documents into OpenSearch using Ollama embeddings!")

# 5. Optional: How you would use ChatOllama later to query it (RAG)
# llm = ChatOllama(model="llama3") 
